In [1]:
import os
import requests
import json

# 1. Access credentials from the environment (loaded by Docker from .env)
client_id = os.getenv("EMT_CLIENT_ID")
passkey = os.getenv("EMT_PASSKEY")

print(f"Credentials check: {'OK' if client_id and passkey else 'MISSING'}")

# 2. Authenticate to get the 'accessToken'
# The EMT API requires a session token for all subsequent requests
login_url = "https://openapi.emtmadrid.es/v1/mobilitylabs/user/login/"
headers = {
    "X-ClientId": client_id,
    "passKey": passkey
}

try:
    login_response = requests.get(login_url, headers=headers)
    login_response.raise_for_status() # Raise error if status is not 200
    
    login_data = login_response.json()
    # Extract the accessToken from the nested structure
    access_token = login_data.get("data", [{}])[0].get("accessToken")
    
    if access_token:
        print("✅ Login successful! Session token retrieved.")
        
        # 3. Test a simple API call: Get info for a specific Stop (e.g., Stop 70 - Sol)
        stop_id = "70"
        arrival_url = f"https://openapi.emtmadrid.es/v1/transport/busemtmad/stops/{stop_id}/arriving/"
        
        arrival_headers = {"accessToken": access_token}
        # The API expects a POST with specific parameters for arrivals
        payload = {
            "statistics": "N",
            "cultureInfo": "EN",
            "Text_StopRequired_YN": "Y",
            "Text_CheckCheckCursor_YN": "B",
            "Text_EstimationsRequired_YN": "Y",
            "Text_IncidencesRequired_YN": "Y"
        }
        
        arrival_response = requests.post(arrival_url, headers=arrival_headers, json=payload)
        arrival_data = arrival_response.json()
        
        print(f"\n--- Real-time arrivals for Stop {stop_id} ---")
        arrivals = arrival_data.get('data', [{}])[0].get('Arriving', [])
        
        if arrivals:
            for bus in arrivals[:5]: # Show first 5 buses
                line = bus.get('line')
                estimate = bus.get('estimateArrive')
                print(f"Bus Line {line}: {estimate // 60} min left.")
        else:
            print("No buses currently arriving or data unavailable.")
            
    else:
        print("❌ Login failed: Could not retrieve accessToken.")

except Exception as e:
    print(f"❌ Error during API test: {e}")

Credentials check: OK
✅ Login successful! Session token retrieved.
❌ Error during API test: Expecting value: line 1 column 1 (char 0)


In [2]:
# 1. Use the access_token we got from the previous successful login
lines_url = "https://openapi.emtmadrid.es/v1/transport/busemtmad/getlistlines/"

headers = {
    "accessToken": access_token,
    "Content-Type": "application/json"
}

# Simple GET request
response = requests.get(lines_url, headers=headers)

if response.status_code == 200:
    lines_data = response.json()
    # Let's see the first 5 lines
    lines = lines_data.get('data', [])
    print(f"✅ Success! Found {len(lines)} lines.")
    for line_info in lines[:5]:
        print(f"Line {line_info.get('line')}: {line_info.get('nameA')} -> {line_info.get('nameB')}")
else:
    print(f"❌ Failed with status: {response.status_code}")
    print(f"Response text: {response.text}")

❌ Failed with status: 404
Response text: <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



In [3]:
# 1. Use the access_token you already retrieved
# Try the v2 endpoint which is more stable for line info
lines_url = "https://openapi.emtmadrid.es/v2/transport/busemtmad/lines/info/20260214/" 

headers = {
    "accessToken": access_token,
    "Content-Type": "application/json"
}

# The EMT v2 lines info often prefers a GET
response = requests.get(lines_url, headers=headers)

if response.status_code == 200:
    data = response.json()
    lines = data.get('data', [])
    print(f"✅ Success! Found {len(lines)} lines.")
    # Show a sample
    for line in lines[:3]:
        print(f"Line {line.get('line')}: {line.get('nameA')} -> {line.get('nameB')}")
else:
    print(f"❌ Error {response.status_code}")
    print(f"Detail: {response.text}")

✅ Success! Found 232 lines.
Line 361: ESTACION DE ATOCHA -> MONCLOA
Line 362: PUERTA DE TOLEDO -> ARGÜELLES
Line 203: ESTACION DE ATOCHA -> AEROPUERTO


In [4]:
stop_id = "70"
# V2 endpoint for arrivals
arrival_url = f"https://openapi.emtmadrid.es/v2/transport/busemtmad/stops/{stop_id}/arrives/"

# V2 requires these specific fields in the JSON body
payload = {
    "cultureInfo": "EN",
    "Text_StopRequired_YN": "Y",
    "Text_CheckCheckCursor_YN": "B",
    "Text_EstimationsRequired_YN": "Y",
    "Text_IncidencesRequired_YN": "Y",
    "DateTime_Referer_ShortRange_String": "2026-02-14T18:30:00",
    "Statistics_Key_Check_String": "N"
}

response = requests.post(arrival_url, headers=headers, json=payload)
print(f"Arrivals Response Status: {response.status_code}")
print(response.json())

Arrivals Response Status: 200
{'code': '00', 'description': ' Data recovered OK  (lapsed: 419 millsecs)', 'datetime': '2026-02-14T18:27:28.184844', 'data': [{'Arrive': [{'line': '20', 'stop': '70', 'isHead': 'False', 'destination': 'PAVONES', 'deviation': 0, 'bus': 3467, 'geometry': {'type': 'Point', 'coordinates': [-3.6942708975506116, 40.41903341211236]}, 'estimateArrive': 0, 'DistanceBus': 0, 'positionTypeBus': '0'}, {'line': '52', 'stop': '70', 'isHead': 'False', 'destination': 'SANTAMARCA', 'deviation': 0, 'bus': 2444, 'geometry': {'type': 'Point', 'coordinates': [-3.6971340057587567, 40.41891707858794]}, 'estimateArrive': 0, 'DistanceBus': 0, 'positionTypeBus': '0'}, {'line': '1', 'stop': '70', 'isHead': 'False', 'destination': 'PROSPERIDAD', 'deviation': 0, 'bus': 5720, 'geometry': {'type': 'Point', 'coordinates': [-3.700573698747189, 40.419842222315175]}, 'estimateArrive': 144, 'DistanceBus': 333, 'positionTypeBus': '0'}, {'line': '2', 'stop': '70', 'isHead': 'False', 'destinat

In [10]:
import datetime
import requests

# 1. Credentials (ensure you have your headers defined from previous cells)
# headers = {"accessToken": "YOUR_TOKEN_HERE"} 

# 2. Fix the Date Format: APIs hate slashes "/" in parameters. 
# We use YYYYMMDD (Standard)
target_date = datetime.datetime.now().strftime("%Y%m%d")
line_id = "27"

# 3. Use the V2 Endpoint for Line Information
# This endpoint gives the "Topology" (Head, Tail, and basic schedule info)
url = f"https://openapi.emtmadrid.es/v2/transport/busemtmad/lines/info/{target_date}/"

print(f"Requesting: {url}")

try:
    response = requests.get(url, headers=headers)
    
    # Check if successful
    if response.status_code == 200:
        data = response.json()
        
        # The API returns ALL lines, so we must find Line 27 inside the list
        all_lines = data.get('data', [])
        
        # Search for our specific line
        # We use a "Generator Expression" to find the first match
        my_line = next((item for item in all_lines if item["label"] == line_id), None)
        
        if my_line:
            print(f"✅ FOUND LINE {line_id}!")
            print(f"   From: {my_line.get('nameA')}")
            print(f"   To:   {my_line.get('nameB')}")
            print(f"   Start Time: {my_line.get('startTime')}")
            print(f"   End Time:   {my_line.get('stopTime')}")
            print(f"   Min Frequency: {my_line.get('minimumFrequency')}")
            print(f"   Max Frequency: {my_line.get('maximumFrequency')}")
        else:
            print(f"❌ Line {line_id} not found in the response list.")
            
    else:
        print(f"❌ Error {response.status_code}: {response.text}")

except Exception as e:
    print(f"🔥 Critical Error: {e}")
    

Requesting: https://openapi.emtmadrid.es/v2/transport/busemtmad/lines/info/20260214/
✅ FOUND LINE 27!
   From: EMBAJADORES
   To:   PLAZA CASTILLA
   Start Time: None
   End Time:   None
   Min Frequency: None
   Max Frequency: None


In [8]:
import datetime

# 1. Fix the Date Format (YYYYMMDD is the standard for APIs)
today = datetime.datetime.now().strftime("%Y%m%d") 
line_id = "27"

# 2. Use the v2 endpoint which we know works better
# Notice we don't put the date in the URL for this specific endpoint in v2, 
# or we use the specific "lines/info" one.
# Let's try the specific "TimeTable" endpoint if available, otherwise "Line Info".

# Option A: Get general line info (which contains frequency/schedule data)
schedule_url = f"https://openapi.emtmadrid.es/v2/transport/busemtmad/lines/info/{today}/"

# 3. We must filter for our specific line manually after fetching, 
# because v2 often gives all lines at once.
response = requests.get(schedule_url, headers=headers)

print(f"Schedule Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    all_lines = data.get('data', [])
    
    # Filter for Line 27
    line_27_data = next((item for item in all_lines if item["label"] == line_id), None)
    
    if line_27_data:
        print(f"✅ Found Schedule for Line {line_id}")
        print(f"From: {line_27_data.get('startTime')}")
        print(f"To: {line_27_data.get('stopTime')}")
        print(f"Min Frequency: {line_27_data.get('minimumFrequency')} mins")
        print(f"Max Frequency: {line_27_data.get('maximumFrequency')} mins")
    else:
        print(f"❌ Line {line_id} not found in today's schedule.")
else:
    print("❌ API Error")

Schedule Status: 200
✅ Found Schedule for Line 27
From: None
To: None
Min Frequency: None mins
Max Frequency: None mins


In [9]:
# Let's see the keys explicitly
print(line_27_data.keys())
# And the raw data to spot the correct field names
print(line_27_data)

dict_keys(['line', 'label', 'nameA', 'nameB', 'group', 'startDate', 'endDate', 'longLine1', 'longLine2', 'order', 'color'])
{'line': '027', 'label': '27', 'nameA': 'EMBAJADORES', 'nameB': 'PLAZA CASTILLA', 'group': '110', 'startDate': '14/02/2026', 'endDate': '14/02/2026', 'longLine1': 8294, 'longLine2': 7664, 'order': 3, 'color': '0072ce'}


In [7]:
# Experiment 3: BiciMAD Stations
# Note: BiciMAD usually has its own endpoint structure
bicimad_url = "https://openapi.emtmadrid.es/v1/transport/bicimad/stations/"

response = requests.get(bicimad_url, headers=headers)
print(f"BiciMAD Status: {response.status_code}")

if response.status_code == 200:
    stations = response.json().get('data', [])
    print(f"✅ Found {len(stations)} BiciMAD stations.")
    # Let's look at the first one
    s = stations[0]
    print(f"Station: {s.get('name')} | Bikes Available: {s.get('dock_bikes')} / {s.get('total_bases')}")

BiciMAD Status: 200
✅ Found 633 BiciMAD stations.
Station: 5 - Fuencarral | Bikes Available: 6 / 27
